
# 1. Dataset

`Dataset` is an abstract base class in `torch.utils.data`. It defines a contract, not behavior — you subclass it and implement the details yourself.

```python
from torch.utils.data import Dataset
import torch

class MyDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
```

### The two required methods

| Method | Returns | Purpose |
|---|---|---|
| `__len__(self)` | a single **integer** | total number of items in the dataset |
| `__getitem__(self, idx)` | **the item** at position `idx` (not the index itself) | given an index, produce/return the corresponding item |

These are Python's special (dunder) methods — the same mechanism behind `len(x)` and `x[i]` for lists. Implementing them makes your object behave like an indexable sequence, which is exactly what the `DataLoader` needs to interact with it (it never needs to know what's inside — just that it can call `len(dataset)` and `dataset[idx]`).

```python
ds = MyDataset(x, y)

len(ds)     # calls __len__() → e.g. 100
ds[5]       # calls __getitem__(5) → e.g. (tensor([...]), tensor(1))
```

### Where the data actually lives

Nothing about `Dataset` forces you to load everything into memory — that depends entirely on how you write `__init__` and `__getitem__`.

**Small dataset (fits comfortably in RAM):** load everything up front in `__init__`.

```python
class SmallDataset(Dataset):
    def __init__(self, x, y):
        self.x = x   # already a full tensor in memory
        self.y = y

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]   # just indexes what's already in RAM
```

**Large dataset (doesn't fit, or you want parallel loading):** store only lightweight references (paths, metadata) in `__init__`, and do the heavy work (reading a file, decoding an image) inside `__getitem__`, on demand.

```python
class ImageDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths     # just a list of strings, cheap
        self.labels = labels

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx])   # loaded only now, one at a time
        return image, self.labels[idx]
```

The reason this matters: when the `DataLoader` uses `num_workers > 0`, it calls `__getitem__` in parallel worker processes. If the expensive work happens inside `__getitem__`, it parallelizes automatically. If everything were already loaded in `__init__`, there'd be nothing left to parallelize — and large datasets might not fit in memory at all.

### Common mistake

`__len__` must return a plain integer — not a tuple:

```python
def __len__(self):
    return len(self.x), len(self.y)   # ❌ wrong: returns a tuple (10, 10)

def __len__(self):
    return len(self.x)                # ✅ correct: a single int
```

---


In [22]:
from torch.utils.data import Dataset
import torch

class MeuDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]    


x, y = torch.rand(10, ), torch.rand(10,)
output  = MeuDataset(x, y)

In [28]:
print(f'Len of dataset: {len(output)}')
print(f'{output[0]}')

Len of dataset: 10
(tensor(0.6404), tensor(0.9204))



## TensorDataset

A ready-made `Dataset` subclass for the common case where your data is **already** a set of tensors and you don't need any custom loading logic (no file reading, no transforms).

```python
from torch.utils.data import TensorDataset
import torch

x = torch.rand(100, 10)
y = torch.rand(100, 1)

ds = TensorDataset(x, y)

len(ds)    # → 100
ds[5]      # → (x[5], y[5])
```

It implements `__len__` and `__getitem__` for you — essentially the same code you'd write by hand for a simple case like `MyDataset` above. It accepts any number of tensors, as long as they all agree on the size of the first dimension:

```python
ds = TensorDataset(x, y, z)   # works with 2, 3, or more tensors
ds[5]                         # → (x[5], y[5], z[5])
```

In [35]:
from torch.utils.data import TensorDataset

x, y = torch.rand(10, 2), torch.rand(10, 1)
dataset = TensorDataset(x, y)

print(dataset[0])


(tensor([0.7598, 0.1913]), tensor([0.3032]))
